# 34장 · 요약을 한 번 더 쓸지 결정하는 그래프

전광판 담당자가 길이를 넘긴 원고를 반려했습니다. 같은 요약을 계속 요청하면 출력이 계속 길거나 호출 횟수가 늘어날 수 있습니다. 초안 한 번, 수정 기회 한 번을 주고 그 뒤에는 편집자가 확인하도록 경로를 설계합니다.

짝 활동에는 [실습 수첩](pre-mid/documents/graph/workshop.md)을 사용합니다. 과제 아래에서 풀이를 펼칠 수 있습니다.

## 통과·재작성·인계의 세 경우

상한은 18자입니다. 길이가 18자 이하면 첫 시도든 둘째 시도든 종료합니다. 19자 이상이고 처음 쓴 답이면 다시 씁니다. 같은 문제가 두 번째에도 남으면 편집자에게 넘깁니다. 횟수는 모델을 부른 직후 증가시켜야 “두 번 썼다”의 의미가 일정합니다.

교재 ZIP을 풀고 그 폴더에서 Jupyter를 엽니다. Python 3.12 커널에서 필요한 패키지가 없으면 새 셀에서 `%pip install "langgraph>=1,<2" openai streamlit pypdf python-dotenv`를 실행하고 커널을 다시 시작하세요. 실습 자료 폴더와 노트북은 함께 보관합니다.

```mermaid
flowchart TD
  A[초안 작성] --> B{18자 이내인가}
  B -->|예| C[게시 가능]
  B -->|아니요| D{수정 기회가 남았나}
  D -->|예| E[다시 작성]
  E --> B
  D -->|아니요| F[편집자 확인]
```

In [1]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class Draft(TypedDict, total=False):
    source: str
    text: str
    tries: int
    reason: str

def choose(state):
    if len(state["text"]) <= 18:
        return "publish"
    if state["tries"] < 2:
        return "revise"
    return "review"

print(choose({"text": "가" * 18, "tries": 1}))
print(choose({"text": "가" * 19, "tries": 1}))
print(choose({"text": "가" * 19, "tries": 2}))

publish
revise
review


분기 함수는 실행할 작업의 이름을 반환합니다. 상태를 고치는 노드와 역할이 다릅니다. 길이를 통과한 문장도 의미가 맞는지는 사람이 읽습니다.

## 글을 쓰는 방법과 경로를 따로 시험하기

같은 연결을 실제 모델과 연습용 답으로 각각 실행합니다. 모델 답이 우연히 짧게 나오더라도 재작성과 상한을 시험할 수 있도록, 먼저 정해 둔 문장을 공급합니다. 이때 나오는 문장은 수업용 입력입니다.

In [2]:
def build_editor(write):
    def first(state):
        return {"text": write(state["source"], ""), "tries": 1}

    def revise(state):
        return {"text": write(state["source"], state["text"]), "tries": state["tries"] + 1}

    plan = StateGraph(Draft)
    plan.add_node("first", first)
    plan.add_node("revise", revise)
    plan.add_node("publish", lambda state: {"reason": "길이 통과"})
    plan.add_node("review", lambda state: {"reason": "수정 기회 소진"})
    plan.add_edge(START, "first")
    for name in ["first", "revise"]:
        plan.add_conditional_edges(name, choose, ["publish", "revise", "review"])
    plan.add_edge("publish", END)
    plan.add_edge("review", END)
    return plan.compile()

def rehearsal(source, previous):
    return "동문 폐쇄, 서문 이용" if previous else "오늘 오후 점검으로 동문 출입이 어려우니 서문으로 들어와 주세요."

editor = build_editor(rehearsal)
notice = "오후 2시부터 4시까지 동문을 닫습니다. 방문자는 서문으로 들어오세요."
for update in editor.stream({"source": notice}, stream_mode="updates"):
    print(update)

{'first': {'text': '오늘 오후 점검으로 동문 출입이 어려우니 서문으로 들어와 주세요.', 'tries': 1}}
{'revise': {'text': '동문 폐쇄, 서문 이용', 'tries': 2}}
{'publish': {'reason': '길이 통과'}}


첫 초안이 길어 revise가 실행됩니다. 두 번째 문장은 길이를 통과하지만 폐쇄 시간이 빠졌습니다. 프로그램의 길이 기준을 통과했다는 기록과 실제로 게시해도 된다는 업무 판단을 따로 확인하세요.

## 실제 모델로 원고 보내기

8장에서 쓴 API 키로 실행합니다. 재작성 요청에는 원문, 직전 초안, 보존할 시각과 출입구를 함께 넣습니다. 초안이 이미 짧으면 두 번째 호출은 일어나지 않습니다. 출력은 실행 때마다 달라질 수 있습니다.

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# 8장에서 준비한 .env를 읽습니다. 키는 출력하지 않습니다.
load_dotenv()
use_live = bool(os.getenv("OPENAI_API_KEY"))
model_name = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
if use_live:
    client = OpenAI()

In [4]:
def write_with_model(source, previous):
    # 재작성 때도 원문을 함께 보여 줘서 생략된 정보를 다시 찾을 수 있게 합니다.
    content = f"원문\n{source}\n직전 초안\n{previous}"
    return client.responses.create(model=model_name, store=False,
        instructions="공백 포함 18자 이내 한국어 게시문을 쓰세요. 시간과 출입구를 보존하세요.",
        input=content).output_text.strip()

if use_live:
    for update in build_editor(write_with_model).stream({"source": notice}, stream_mode="updates"):
        print(update)
else:
    print("실제 모델 실행은 API 키를 준비한 뒤 이 셀에서 확인하세요.")

실제 모델 실행은 API 키를 준비한 뒤 이 셀에서 확인하세요.


실제 결과에서 노드 이름을 순서대로 적고 동문, 서문, 시각을 확인하세요. 모델이 원문에 없는 날짜를 덧붙였다면 길이와 별개로 게시를 보류할 이유가 됩니다.

## 문장이 짧아졌는데도 발송할 수 없는 이유

편집자가 반드시 남기라고 표시한 두 항목이 있습니다. 하나는 출입이 막히는 시간, 하나는 사용할 대체 출입구입니다. “서문으로 들어오세요”는 화면 한 줄에 들어가지만 동문 폐쇄가 몇 시부터인지 알려 주지 않습니다. 독자가 누구이고 어떤 사실을 알아야 움직일 수 있는지 먼저 정해야, 두 번째 작성에서 무엇을 고칠지 모델에게 알려 줄 수 있습니다.

In [5]:
brief = "14~16시 동문폐쇄·서문이용"
short = "서문으로 들어오세요"
for note in (brief, short):
    print(note, len(note), "시간 있음:", "14~16시" in note, "대체 문 있음:", "서문" in note)

14~16시 동문폐쇄·서문이용 16 시간 있음: True 대체 문 있음: True
서문으로 들어오세요 10 시간 있음: False 대체 문 있음: True


두 문장 모두 길이를 통과해도 시각은 한 문장에만 남습니다. 이 셀은 한 가지 시각 표기를 검사하는 예입니다. “오후 두 시부터 네 시”로 맞게 써도 앞의 문자열 검사는 실패합니다. 실제 편집에서는 서로 다른 표현을 사람이 대조하거나 표현을 통일하는 합의가 필요합니다.

## 실습 · 편집자에게 인계되는 입력 만들기

모델이 언제나 19자 이상을 쓴다는 조건으로 상한을 시험하세요. 방문 순서와 tries를 증거로 제출하고, 세 번째 호출이 왜 없는지 분기 함수의 조건을 짚으세요.

In [6]:
# 위 조건을 만족하는 코드를 이 셀에 작성하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>long_editor = build_editor(lambda source, previous: &quot;가&quot; * 19)
for update in long_editor.stream({&quot;source&quot;: notice}, stream_mode=&quot;updates&quot;):
    print(update)</code></pre><p>first, revise, review가 각각 한 번씩 나옵니다. 두 번째 작성 후 tries가 2이므로 review로 이동하고 END에서 끝납니다.</p></details>

## 실습 · 원문의 숫자를 지키는 편집 회의

“내일 10시까지 2층 자료실로 제출”을 새 원문으로 줍니다. 실제 요약을 한 번 요청해 보세요. 게시 가능한 길이인지, 시간과 층수가 보존됐는지 따로 기록합니다. 초안 한 건으로 다시 쓸 조건을 제안하세요.

In [7]:
# 위 조건을 만족하는 코드를 이 셀에 작성하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>candidate = &quot;내일 10시까지 자료실로 제출&quot;
for phrase in (&quot;10시&quot;, &quot;2층&quot;):
    print(phrase, phrase in candidate)
print(&quot;글자 수&quot;, len(candidate))</code></pre><p>이 예시 초안은 시간은 남겼지만 2층을 빠뜨렸습니다. 원문에서 필수 정보를 먼저 표시하고, 길이를 통과한 뒤에도 누락된 필수 정보가 있으면 편집자 검토로 보낼 수 있습니다.</p></details>

## 전이 과제 · 첫 게시문이 짧으면 모델을 몇 번 부를까

모델 요금을 확인하는 동료가 “모든 원고에 초안과 재작성 비용이 드나요?”라고 묻습니다. 그래프에 노드가 두 개 있다고 실제 두 번 실행된다고 결론내릴 수 없습니다. 어떤 입력이 어떤 간선을 탔는지 실행을 한 번 관찰해야 합니다. 같은 원고라도 첫 응답 길이가 달라질 수 있으니 모델 동작과 경로 검증을 구분합니다.

게시 가능한 초안, 수정 뒤에만 게시 가능한 초안, 두 번 모두 긴 초안을 각각 하나씩 준비합니다. 실제 호출 대신 내가 정한 문장을 반환하게 하고 호출 횟수를 세세요. 결과 검사를 위해 다시 invoke한 것도 별도의 호출임을 이유와 함께 설명하세요.

In [8]:
# 먼저 예상 결과를 적고, 위 상황을 재현할 셀을 작성하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>for outputs in [[&quot;서문 이용&quot;], [&quot;가&quot; * 19, &quot;서문 이용&quot;], [&quot;가&quot; * 19, &quot;나&quot; * 19]]:
    replies = iter(outputs)
    graph = build_editor(lambda source, previous: next(replies))
    result = graph.invoke({&quot;source&quot;: notice})
    print(result[&quot;tries&quot;], result[&quot;reason&quot;])</code></pre><p>호출 횟수는 각각 1, 2, 2입니다. 첫 경우에는 revise가 실행되지 않습니다. 실제 서비스에서 비용을 셀 때는 저장된 그 실행의 기록을 읽습니다. invoke로 다시 확인하면 답이 변하거나 비용이 추가될 수 있습니다.</p></details>

다음 장에서는 한 번의 실행이 끝난 뒤에도 다음 질문에 남아야 하는 정보를 다룹니다. [조건부 간선과 상태](https://docs.langchain.com/oss/python/langgraph/graph-api)